# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/surajprai2111-ui/ML-assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Why I chose these models

I will compare three classification models: Logistic Regression, Decision Tree and Random Forest.

- **Logistic Regression** is my starting point because it is relatively simple and gives me a useful reference.
- **Decision Tree** can capture non-linear patterns and is easier to inspect than a large ensemble.
- **Random Forest** combines multiple decision trees, so I can check whether this improves the results.

I will compare all three with my ML-07 baseline using the same eligible pages, client-grouped train/test split and Average Precision metric.

One limitation is that my target is based on the existing `trend_direction` field. The model will identify patterns associated with that observed label; it will not prove that a page will decline in the future or that refreshing it will improve performance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix
)

# Load the same starter dataset used in ML-07
url = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(url)

print("Original dataset shape:", df.shape)

# Keep the same eligible population as ML-07
eligible = df.loc[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
].drop_duplicates(subset="content_id").copy()

# Create the target from the observed trend label
eligible["is_declining"] = (
    eligible["trend_direction"] == "down"
).astype(int)

# Candidate numeric features
numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Candidate categorical features
categorical_candidates = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

# Use only columns that actually exist in this dataset
numeric_features = [
    col for col in numeric_candidates
    if col in eligible.columns
]

categorical_features = [
    col for col in categorical_candidates
    if col in eligible.columns
]

feature_columns = numeric_features + categorical_features

# Confirm that IDs and target-derived fields are not model features
forbidden_features = {
    "content_id",
    "client_id",
    "is_declining",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

assert not (set(feature_columns) & forbidden_features)

print("Eligible pages:", len(eligible))
print("Unique content IDs:", eligible["content_id"].nunique())
print("Unique clients:", eligible["client_id"].nunique())
print("Declining label count:", eligible["is_declining"].sum())
print(
    "Declining label rate (%):",
    round(eligible["is_declining"].mean() * 100, 2)
)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Total candidate features:", len(feature_columns))

Original dataset shape: (30000, 44)
Eligible pages: 30000
Unique content IDs: 30000
Unique clients: 32
Declining label count: 16262
Declining label rate (%): 54.21
Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
Total candidate features: 26


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will split the data by client using an 80/20 train-test split. This means that pages belonging to the same client will not appear in both the training and test sets.

I am using a client-grouped split because pages from one client may share similar characteristics. Keeping clients separate gives a more honest check of whether the model works on clients it has not seen during training.

This split is not time-aware. The starter dataset contains one row per content item with summary metrics, so this evaluation checks generalisation to unseen clients rather than predicting a future month. I will use the same split for all models and the baseline comparison.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



from sklearn.model_selection import GroupShuffleSplit

# Check that the required data is available
required_columns = ["client_id", "is_declining"]

missing_columns = [
    col for col in required_columns
    if col not in eligible.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Split by client: 80% training and 20% testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        eligible,
        y=eligible["is_declining"],
        groups=eligible["client_id"]
    )
)

train_df = eligible.iloc[train_idx].copy()
test_df = eligible.iloc[test_idx].copy()

# Verify the split
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Clients in both sets:", len(train_clients & test_clients))

print("\nTraining target distribution (%):")
print(
    (train_df["is_declining"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nTest target distribution (%):")
print(
    (test_df["is_declining"].value_counts(normalize=True) * 100)
    .round(2)
)



Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Clients in both sets: 0

Training target distribution (%):
is_declining
1    55.01
0    44.99
Name: proportion, dtype: float64

Test target distribution (%):
is_declining
1    51.1
0    48.9
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model comparison

I will compare Logistic Regression, Decision Tree and Random Forest with the ML-07 baseline. I will use the same client-grouped training and test sets for all three models.

I will exclude IDs and trend fields used to create the declining label from the model features. I will compare Average Precision because it evaluates how well the models rank declining pages above other pages. I will also review the errors before choosing a model.

The declining label is based on the observed trend in the starter dataset. Therefore, the results measure how well the models identify this label, not whether a content refresh will improve future performance.

The first comparison shows that Random Forest performed best, with an Average Precision of 0.6109. Decision Tree scored 0.5802, while Logistic Regression scored 0.5763. The ML-07 baseline scored 0.5008.

All three trained models performed better than the baseline on the same client-grouped test set. Random Forest is my current preferred model because it achieved the highest Average Precision. This is an initial comparison, however, and does not prove that the model will perform equally well on future data or that refreshing a page will improve its performance.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




# This cell trains three models and compares them with the ML-07 baseline.
# The same train_df and test_df are used for every comparison.

# 1. Prepare the baseline score using the ML-07 formula
baseline_df = eligible.copy()

baseline_df["visibility_score"] = (
    np.log1p(baseline_df["impressions_90d"])
    / np.log1p(baseline_df["impressions_90d"].max())
) * 100

baseline_df["freshness_risk_score"] = (
    baseline_df["days_since_last_update"]
    .clip(lower=0, upper=365) / 365
) * 100

baseline_df["freshness_risk_score"] = (
    baseline_df["freshness_risk_score"].fillna(0)
)

position = baseline_df["avg_position"]

baseline_df["position_opportunity_score"] = np.where(
    position.between(1, 20),
    ((position - 1) / 19) * 100,
    0
)

baseline_df["depth_gap_score"] = np.select(
    [
        baseline_df["word_count"].between(1, 599),
        baseline_df["word_count"].between(600, 1199)
    ],
    [100, 50],
    default=0
)

baseline_df["baseline_action_score"] = (
    0.40 * baseline_df["visibility_score"]
    + 0.30 * baseline_df["freshness_risk_score"]
    + 0.25 * baseline_df["position_opportunity_score"]
    + 0.05 * baseline_df["depth_gap_score"]
)

# Match baseline scores to the existing test rows using content_id.
baseline_lookup = baseline_df.set_index(
    "content_id"
)["baseline_action_score"]

baseline_scores = test_df["content_id"].map(baseline_lookup)

assert baseline_scores.notna().all(), (
    "Some test pages do not have a baseline score."
)

# 2. Prepare training and test data
X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["is_declining"]
y_test = test_df["is_declining"]

# 3. Handle missing values and categorical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# 4. Define three models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=50,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )
}

# 5. Evaluate the baseline and trained models using Average Precision
results = [{
    "Model": "ML-07 Baseline",
    "Average Precision": average_precision_score(
        y_test, baseline_scores
    )
}]

trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    probabilities = pipeline.predict_proba(X_test)[:, 1]

    ap = average_precision_score(y_test, probabilities)

    results.append({
        "Model": name,
        "Average Precision": ap
    })

    trained_models[name] = pipeline

# 6. Display the comparison
comparison = pd.DataFrame(results)

comparison = comparison.sort_values(
    "Average Precision",
    ascending=False
).reset_index(drop=True)

comparison["Average Precision"] = (
    comparison["Average Precision"].round(4)
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Baseline scores matched:", baseline_scores.notna().sum())

print("\nModel comparison (higher Average Precision is better):")
print(comparison.to_string(index=False))


Training rows: 23837
Test rows: 6163
Baseline scores matched: 6163

Model comparison (higher Average Precision is better):
              Model  Average Precision
      Random Forest             0.6109
      Decision Tree             0.5802
Logistic Regression             0.5763
     ML-07 Baseline             0.5008


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I will inspect the Random Forest model because it achieved the highest Average Precision in my initial comparison. I will check its false positives and false negatives to understand where it makes mistakes.

I will also check which features the model relies on most. These results can help explain its decisions, but feature importance does not prove that a feature causes content to decline. Since the target comes from the existing `trend_direction` label, the analysis describes patterns associated with that label rather than confirmed future decline.

### Error analysis and interpretation

The Random Forest achieved an Average Precision of **0.6109**, the highest among the three models tested. However, it still makes a substantial number of classification errors. The precision, recall and confusion matrix reported below use a probability threshold of 0.5. A different threshold could change the balance between false positives and false negatives.

The confusion matrix shows **1,502 false positives** and **1,040 false negatives**. False positives are pages predicted as declining even though their observed label is not declining. False negatives are pages labelled as declining that the model misses. In a content-review workflow, false positives may waste review time, while false negatives may cause potentially important pages to be overlooked.

The model's recall for the declining class is **0.67**, meaning it identifies about 67% of the pages labelled as declining in the test set. Its precision for this class is **0.58**, meaning about 58% of the pages it predicts as declining actually have the observed declining label.

The most important features in the Random Forest are `avg_position` (0.1169), `impressions_90d` (0.1154), `days_with_impressions` (0.1122), and `content_age_days` (0.0916). This suggests that the model relies heavily on search position, visibility, how consistently pages receive impressions, and page age when identifying the existing label.

These results are useful, but they do not prove that these factors cause a page to decline. The target is derived from the existing `trend_direction` field, so the model learns patterns associated with that label rather than predicting a guaranteed future decline. A human reviewer should check the page context before recommending a refresh.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




# Inspect errors and feature importance for the best model.

from sklearn.metrics import confusion_matrix, classification_report

# 1. Get Random Forest predictions on the existing test set
best_model = trained_models["Random Forest"]

rf_probabilities = best_model.predict_proba(X_test)[:, 1]
rf_predictions = (rf_probabilities >= 0.5).astype(int)

# 2. Check the prediction errors
print("Confusion matrix:")
print(confusion_matrix(y_test, rf_predictions))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        rf_predictions,
        target_names=["Not declining", "Declining"],
        zero_division=0
    )
)

# 3. Display examples of false positives and false negatives
error_review = test_df[
    ["content_id", "client_id", "is_declining"]
].copy()

error_review["predicted_declining"] = rf_predictions
error_review["predicted_probability"] = rf_probabilities

false_positives = error_review[
    (error_review["is_declining"] == 0)
    & (error_review["predicted_declining"] == 1)
]

false_negatives = error_review[
    (error_review["is_declining"] == 1)
    & (error_review["predicted_declining"] == 0)
]

print("\nFalse positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFalse-positive examples:")
print(false_positives.head(5).to_string(index=False))

print("\nFalse-negative examples:")
print(false_negatives.head(5).to_string(index=False))

# 4. Check which original features matter most
rf_pipeline = best_model
preprocessor_fitted = rf_pipeline.named_steps["preprocessor"]
rf_classifier = rf_pipeline.named_steps["model"]

feature_names = preprocessor_fitted.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_classifier.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 15 features by Random Forest importance:")
print(
    importance_df.head(15).to_string(index=False)
)


Confusion matrix:
[[1512 1502]
 [1040 2109]]

Classification report:
               precision    recall  f1-score   support

Not declining       0.59      0.50      0.54      3014
    Declining       0.58      0.67      0.62      3149

     accuracy                           0.59      6163
    macro avg       0.59      0.59      0.58      6163
 weighted avg       0.59      0.59      0.58      6163


False positives: 1502
False negatives: 1040

False-positive examples:
          content_id         client_id  is_declining  predicted_declining  predicted_probability
content_a5a2fbc76336 client_8527a891e2             0                    1               0.845985
content_72c5c2d73e5a client_4e07408562             0                    1               0.801250
content_bce275871a25 client_f369cb89fc             0                    1               0.645963
content_dcebfd222b10 client_f369cb89fc             0                    1               0.534929
content_685de0e3b7cb client_f369cb89fc    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.